# Init

In [0]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("gold.dim_products")

# Transformation Logic

In [0]:
query = """
SELECT
    ROW_NUMBER() OVER (ORDER BY crm_p.start_date, crm_p.product_key) AS product_surrogate_key,
    crm_p.product_id,
    crm_p.product_key,
    crm_p.product_name,
    crm_p.category_id,
    erp_p_cat.category,
    erp_p_cat.subcategory,
    erp_p_cat.maintenance_flag,
    crm_p.product_line,
    crm_p.start_date
FROM silver.crm_products crm_p
LEFT JOIN silver.erp_product_category erp_p_cat
    ON crm_p.category_id = erp_p_cat.category_id
"""
df = spark.sql(query)

# Write into Gold Table

In [0]:
try:
    df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_products")
except Exception as e:
    logger.error(f"Failed to write Gold table: {e}")
    raise